In [7]:
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import torch

data = [
    ("الخدمة ممتازة جدا شكرا", 2),
    ("انا مبسوط جدا من التعامل", 2),
    ("منتج رائع انصح بيه", 2),
    ("شكرا على السرعة", 2),
    ("التوصيل كان سريع ومحترم", 2),
    ("الخدمة عادية مش وحشة", 1),
    ("مقبول بس ممكن يتحسن", 1),
    ("عادي يعني", 1),
    ("مش احسن حاجة بس ماشي", 1),
    ("السعر غالي شوية", 1),
    ("انا زعلان جدا الخدمة سيئة", 0),
    ("الاوردر اتأخر 3 ايام", 0),
    ("محدش بيرد عليا", 0),
    ("المنتج وصل بايظ", 0),
    ("تجربة سيئة مش هكررها", 0),
]
data = data * 10
df = pd.DataFrame(data, columns=['text', 'label'])

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

model_name = "asafaya/bert-base-arabic"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)

def tokenize(batch):
    return tokenizer(batch['text'], padding=True, truncation=True, max_length=128)

from datasets import Dataset
train_ds = Dataset.from_pandas(train_df)
test_ds = Dataset.from_pandas(test_df)
train_ds = train_ds.map(tokenize, batched=True)
test_ds = test_ds.map(tokenize, batched=True)

def compute_metrics(p):
    preds = p.predictions.argmax(-1)
    return {'accuracy': accuracy_score(p.label_ids, preds)}

args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    logging_steps=10,
    save_strategy="no"
)

trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=test_ds, compute_metrics=compute_metrics)
trainer.train()
print(trainer.evaluate())

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: asafaya/bert-base-arabic
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/120 [00:00<?, ? examples/s]

Map:   0%|          | 0/30 [00:00<?, ? examples/s]

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
10,0.932280
20,0.314615


Training Loss,Validation Loss,Step,Accuracy
0.314615,0.083348,24,1.000000


{'eval_loss': 0.08334825932979584, 'eval_accuracy': 1.0}


In [8]:
def predict_comment(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    import torch
    with torch.no_grad():
        outputs = model(**inputs)
    pred = outputs.logits.argmax(-1).item()
    labels = {0: "سلبي 😡", 1: "محايد 😐", 2: "ايجابي 😍"}
    return labels[pred]

print(predict_comment("الخدمة وحشة جدا ومحدش بيرد"))
print(predict_comment("شكرا جدا انتو ناس محترمة"))
print(predict_comment("الاوردر اتأخر 3 ايام"))
print(predict_comment("التوصيل كان سريع"))

سلبي 😡
ايجابي 😍
سلبي 😡
ايجابي 😍


In [9]:
import re

responses = {
    "بكام": "السعر 350 جنيه، والتوصيل حسب محافظتك.",
    "سعر": "السعر 350 جنيه، والتوصيل حسب محافظتك.",
    "شحن": "الشحن بياخد من 2 لـ 4 أيام عمل.",
    "توصيل": "الشحن بياخد من 2 لـ 4 أيام عمل.",
    "يوصل": "الشحن بياخد من 2 لـ 4 أيام عمل.",
    "دفع": "تقدر تدفع كاش عند الاستلام أو فودافون كاش.",
    "ادفع": "تقدر تدفع كاش عند الاستلام أو فودافون كاش.",
    "كاش": "تقدر تدفع كاش عند الاستلام أو فودافون كاش.",
    "استرجاع": "تقدر ترجع أو تستبدل المنتج خلال 14 يوم لو بحالته الأصلية.",
    "ارجع": "تقدر ترجع أو تستبدل المنتج خلال 14 يوم لو بحالته الأصلية.",
    "استبدال": "تقدر ترجع أو تستبدل المنتج خلال 14 يوم لو بحالته الأصلية.",
    "خصم": "عندنا خصم 10% لو طلبت قطعتين أو أكتر، اكتب كود: AHLA10",
    "عرض": "عندنا خصم 10% لو طلبت قطعتين أو أكتر، اكتب كود: AHLA10",
    "عنوان": "فرعنا في القاهرة، شارع مصطفى النحاس، ومتاح توصيل لكل المحافظات.",
    "مكان": "فرعنا في القاهرة، شارع مصطفى النحاس، ومتاح توصيل لكل المحافظات.",
    "فين": "فرعنا في القاهرة، شارع مصطفى النحاس، ومتاح توصيل لكل المحافظات.",
    "مواعيد": "شغالين كل يوم من 10 صباحاً لـ 10 مساءً.",
    "فاتحين": "شغالين كل يوم من 10 صباحاً لـ 10 مساءً.",
    "سلام": "أهلاً بيك! نورتنا، تحب أساعدك في إيه؟",
    "اهلا": "أهلاً بيك! نورتنا، تحب أساعدك في إيه؟",
    "ازيك": "أهلاً بيك! نورتنا، تحب أساعدك في إيه؟",
}

def chatbot_reply(message):
    message = re.sub(r'[^\w\s]', '', message)
    for key, reply in responses.items():
        if key in message:
            return reply
    return "معلش مش فاهم سؤالك، ممكن توضح؟ جرب تسأل عن السعر أو الشحن أو الدفع."

print(chatbot_reply("السلام عليكم"))
print(chatbot_reply("بكام؟"))
print(chatbot_reply("التوصيل بياخد قد ايه؟"))
print(chatbot_reply("الدفع ازاي؟"))
print(chatbot_reply("عندكم خصم؟"))
print(chatbot_reply("عايز ارجع المنتج"))
print(chatbot_reply("انتو فين؟"))

أهلاً بيك! نورتنا، تحب أساعدك في إيه؟
السعر 350 جنيه، والتوصيل حسب محافظتك.
الشحن بياخد من 2 لـ 4 أيام عمل.
تقدر تدفع كاش عند الاستلام أو فودافون كاش.
عندنا خصم 10% لو طلبت قطعتين أو أكتر، اكتب كود: AHLA10
تقدر ترجع أو تستبدل المنتج خلال 14 يوم لو بحالته الأصلية.
فرعنا في القاهرة، شارع مصطفى النحاس، ومتاح توصيل لكل المحافظات.
